# Fermi-Dirac Distribution Interactive Workbook
The MB approximation to FD is not “high temperature” by itself. It is the dilute / low-occupation limit:
$$
e^{\beta(\varepsilon-\mu)} \gg 1
$$
or equivalently
$$
\varepsilon-\mu \gg k_B T
$$
In that limit,

$$
f_\text{FD}(\varepsilon)
\approx
\frac{1}{e^{\beta(\varepsilon-\mu)}+1}
\approx
e^{-\beta(\varepsilon-\mu)}
\approx
f_\text{MB}(\varepsilon)
$$

So FD and MB converge when the occupation is small, not automatically when $T$ is large.


For a clean teaching plot, either change the control variable to $\eta = \mu/(k_BT)$, or make $\mu$ much lower than the plotted energy range when comparing MB to FD.

Minimal fix: set the chemical-potential slider mostly below the energy range.

```python
mu_slider = widgets.FloatSlider(
    min=-10.0,
    max=5.0,
    step=0.1,
    value=-2.0,
    description="mu (eV)"
)
```

Then at high temperature and sufficiently negative $\mu$, the gas becomes dilute and FD approaches MB.

The better pedagogical version is to plot against the dimensionless variable

$$
x
=

\beta(\varepsilon-\mu)
$$

Then the distributions are

$$
f_\text{MB}(x)
==============

e^{-x}
$$

$$
f_\text{FD}(x)
==============

\frac{1}{e^x+1}
$$

$$
f_\text{BE}(x)
==============

\frac{1}{e^x-1}
$$

and the classical limit is simply

$$
x \gg 1
$$

That is the cleanest way to show convergence.


In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# # Quantum Occupation Distributions Interactive Workbook

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

%matplotlib inline

kB = 8.617333262145e-5  # eV/K


def fermi_dirac(E, mu, T, kB=kB):
    """
    Fermi-Dirac mean occupation number.

    f_FD(E) = 1 / (exp((E - mu)/(kB T)) + 1)

    Accepts scalars or NumPy arrays.
    """

    E = np.asarray(E)

    if T == 0:
        return np.where(E < mu, 1.0, np.where(E == mu, 0.5, 0.0))

    x = (E - mu) / (kB * T)
    x = np.clip(x, -700, 700)

    return 1.0 / (np.exp(x) + 1.0)


def maxwell_boltzmann(E, mu, T, kB=kB):
    """
    Maxwell-Boltzmann dilute-limit occupation factor.

    f_MB(E) = exp(-(E - mu)/(kB T))

    This is not bounded above by 1.
    """

    E = np.asarray(E)

    if T == 0:
        return np.where(E > mu, 0.0, np.inf)

    x = (E - mu) / (kB * T)
    x = np.clip(x, -700, 700)

    return np.exp(-x)


def bose_einstein(E, mu, T, kB=kB):
    """
    Bose-Einstein mean occupation number.

    f_BE(E) = 1 / (exp((E - mu)/(kB T)) - 1)

    The expression diverges as E approaches mu from above.
    For a physical ideal Bose gas, mu must be below the lowest
    single-particle energy.
    """

    E = np.asarray(E)

    if T == 0:
        return np.zeros_like(E, dtype=float)

    x = (E - mu) / (kB * T)
    x = np.clip(x, -700, 700)

    denom = np.exp(x) - 1.0

    return np.where(denom > 0, 1.0 / denom, np.nan)


plot_output = widgets.Output()


def redraw_workbook(change=None):
    """
    Redraw the occupation distributions for the current slider values.
    """

    mu = mu_slider.value
    T = T_slider.value

    with plot_output:
        plot_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8, 5))

        E_grid = np.linspace(0.0, 5.5, 1000)

        f_FD = fermi_dirac(E_grid, mu, T)
        f_MB = maxwell_boltzmann(E_grid, mu, T)
        f_BE = bose_einstein(E_grid, mu, T)

        ax.plot(
            E_grid,
            f_FD,
            label="Fermi-Dirac",
            linewidth=2.5
        )

        ax.plot(
            E_grid,
            f_MB,
            label="Maxwell-Boltzmann",
            linewidth=2.5,
            linestyle="--"
        )

        ax.plot(
            E_grid,
            f_BE,
            label="Bose-Einstein",
            linewidth=2.5,
            linestyle=":"
        )

        ax.axvline(
            mu,
            linestyle="--",
            linewidth=1.5,
            label=rf"$\mu = {mu:.1f}\ \mathrm{{eV}}$"
        )

        ax.axhline(
            0.5,
            linestyle=":",
            alpha=0.7
        )

        ax.set_xlim(0.0, 5.5)
        ax.set_ylim(-0.05, 2.05)

        ax.set_title("Occupation Distributions", fontsize=14, pad=12)
        ax.set_xlabel("Energy, $E$ (eV)", fontsize=12)
        ax.set_ylabel("Mean occupation, $f(E)$", fontsize=12)

        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9)

        plt.show()


mu_slider = widgets.FloatSlider(
    min=-1.0,
    max=5.0,
    step=0.1,
    value=3.0,
    description="mu (eV)"
)

T_slider = widgets.IntSlider(
    min=0,
    max=2000,
    step=50,
    value=300,
    description="Temp (K)"
)

mu_slider.observe(redraw_workbook, names="value")
T_slider.observe(redraw_workbook, names="value")

redraw_workbook()

widgets.VBox([mu_slider, T_slider, plot_output])

In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# # Occupation Distributions in Dimensionless Form
#
# We define the dimensionless energy variable
#
# $$
# x = \beta(\varepsilon-\mu)
# $$
#
# where
#
# $$
# \beta = \frac{1}{k_BT}
# $$
#
# In terms of $x$, the occupation functions are
#
# $$
# f_\text{MB}(x) = e^{-x}
# $$
#
# $$
# f_\text{FD}(x) = \frac{1}{e^x+1}
# $$
#
# $$
# f_\text{BE}(x) = \frac{1}{e^x-1}
# $$
#
# The Maxwell-Boltzmann limit is the dilute limit:
#
# $$
# x \gg 1
# $$
#
# In that limit,
#
# $$
# e^x \gg 1
# $$
#
# so both Fermi-Dirac and Bose-Einstein statistics approach Maxwell-Boltzmann statistics.

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

%matplotlib inline


def maxwell_boltzmann_x(x):
    """
    Maxwell-Boltzmann occupation factor in dimensionless form.

    f_MB(x) = exp(-x)
    """

    x = np.asarray(x)
    x = np.clip(x, -700, 700)

    return np.exp(-x)


def fermi_dirac_x(x):
    """
    Fermi-Dirac mean occupation number in dimensionless form.

    f_FD(x) = 1 / (exp(x) + 1)
    """

    x = np.asarray(x)
    x = np.clip(x, -700, 700)

    return 1.0 / (np.exp(x) + 1.0)


def bose_einstein_x(x):
    """
    Bose-Einstein mean occupation number in dimensionless form.

    f_BE(x) = 1 / (exp(x) - 1)

    This expression is defined for x > 0.
    It diverges as x approaches 0 from above.
    """

    x = np.asarray(x)
    x = np.clip(x, -700, 700)

    denominator = np.exp(x) - 1.0

    return np.where(denominator > 0, 1.0 / denominator, np.nan)


plot_output = widgets.Output()


def redraw_workbook(change=None):
    """
    Redraw the occupation functions using dimensionless variable x.
    """

    x_min = x_min_slider.value
    x_max = x_max_slider.value
    y_max = y_max_slider.value

    if x_max <= x_min:
        return

    with plot_output:
        plot_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8, 5))

        x_grid = np.linspace(x_min, x_max, 2000)

        f_MB = maxwell_boltzmann_x(x_grid)
        f_FD = fermi_dirac_x(x_grid)
        f_BE = bose_einstein_x(x_grid)

        ax.plot(
            x_grid,
            f_MB,
            label=r"Maxwell-Boltzmann: $e^{-x}$",
            linewidth=2.5
        )

        ax.plot(
            x_grid,
            f_FD,
            label=r"Fermi-Dirac: $\frac{1}{e^x+1}$",
            linewidth=2.5
        )

        ax.plot(
            x_grid,
            f_BE,
            label=r"Bose-Einstein: $\frac{1}{e^x-1}$",
            linewidth=2.5
        )

        ax.axvline(
            0.0,
            linestyle=":",
            linewidth=1.5,
            label=r"$x=0$, where $\varepsilon=\mu$"
        )

        ax.axvspan(
            3.0,
            x_max,
            alpha=0.12,
            label=r"classical dilute region: $x \gg 1$"
        )

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(-0.05, y_max)

        ax.set_title("Maxwell-Boltzmann, Fermi-Dirac, and Bose-Einstein Occupations")
        ax.set_xlabel(r"Dimensionless energy, $x=\beta(\varepsilon-\mu)$")
        ax.set_ylabel(r"Mean occupation, $f(x)$")

        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9)

        plt.show()


x_min_slider = widgets.FloatSlider(
    min=-10.0,
    max=0.0,
    step=0.5,
    value=-5.0,
    description="x min"
)

x_max_slider = widgets.FloatSlider(
    min=1.0,
    max=20.0,
    step=0.5,
    value=10.0,
    description="x max"
)

y_max_slider = widgets.FloatSlider(
    min=1.0,
    max=10.0,
    step=0.5,
    value=3.0,
    description="y max"
)

x_min_slider.observe(redraw_workbook, names="value")
x_max_slider.observe(redraw_workbook, names="value")
y_max_slider.observe(redraw_workbook, names="value")

redraw_workbook()

widgets.VBox(
    [
        x_min_slider,
        x_max_slider,
        y_max_slider,
        plot_output
    ]
)

In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# # Interactive Electronic Carrier Distribution Workbook

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# Ensure inline plotting engine is explicitly configured for the cell context
%matplotlib inline

# %%
def fermi_dirac(E, mu, T):
    """Calculates the Fermi-Dirac occupancy probability."""
    kB = 8.617333262145e-5  # eV/K
    if T == 0:
        return np.piecewise(E, [E < mu, E == mu, E > mu], [1.0, 0.5, 0.0])
    exponent = np.clip((E - mu) / (kB * T), -700, 700)
    return 1.0 / (np.exp(exponent) + 1.0)

def maxwell_boltzmann(E, mu, T):
    """Calculates the classical Maxwell-Boltzmann distribution factor."""
    kB = 8.617333262145e-5  # eV/K
    if T == 0:
        return np.where(E < mu, 1.0, np.where(E == mu, 0.5, 0.0))
    exponent = np.clip((E - mu) / (kB * T), -700, 700)
    return np.exp(-exponent)

def density_of_states(E):
    """Calculates the 3D parabolic density of states g(E) proportional to sqrt(E)."""
    # Returns 0 for negative energy states, smoothly scales with sqrt(E) above 0
    return np.where(E > 0, np.sqrt(E), 0.0)

# %%
# Create a dedicated HTML rendering container for the plot engine
plot_output = widgets.Output()

def redraw_workbook(change=None):
    Ef = Ef_slider.value
    T = T_slider.value
    
    with plot_output:
        plot_output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 5))
        
        # Grid range from 0.0 eV up to 5.5 eV
        E_grid = np.linspace(0.0, 5.5, 1000)
        
        # 1. Compute physical components
        g_E = density_of_states(E_grid)
        f_FD = fermi_dirac(E_grid, Ef, T)
        f_MB = maxwell_boltzmann(E_grid, Ef, T)
        
        # 2. Compute actual electron concentration distributions: n(E) = g(E) * f(E)
        n_FD = g_E * f_FD
        n_MB = g_E * f_MB
        
        # 3. Plot the continuous physical curves
        ax.plot(E_grid, n_FD, label='g(e)f_fd(e)', color='royalblue', lw=2.5)
        ax.plot(E_grid, n_MB, label='g(e)f_mb(e)', color='darkorange', lw=2, linestyle='--')
        ax.plot(E_grid, g_E, label='g(E)', color='dimgray', lw=1.5, linestyle=':')
        
        # Reference markers
        ax.axvline(Ef, color='crimson', linestyle='--', lw=1.5, label=f'$E_f$ = {Ef:.1f} eV')
        
        # 4. HARD-LOCK AXIS VIEWPORTS
        ax.set_xlim(0.0, 5.5)
        # Scaled dynamically to state density max so curves are fully visible without clipping
        ax.set_ylim(-0.05, np.max(g_E) * 1.2)  
        
        # 5. Styling
        ax.set_title('Actual Carrier Distribution: Energy State Density vs. Temperature', fontsize=14, pad=12)
        ax.set_xlabel('Energy, E (eV)', fontsize=12)
        ax.set_ylabel('Number of Occupied States, n(E)', fontsize=12)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
        
        plt.show()

# %%
# Build interface sliders
Ef_slider = widgets.FloatSlider(min=0.5, max=5.0, step=0.1, value=3.0, description='Ef (eV)')
T_slider = widgets.IntSlider(min=50, max=2000, step=50, value=300, description='Temp (K)')

# Attach observer threads
Ef_slider.observe(redraw_workbook, names='value')
T_slider.observe(redraw_workbook, names='value')

# Trigger layout render instantly on cell load
redraw_workbook()

# Return clean widget framework configuration
widgets.VBox([Ef_slider, T_slider, plot_output])


In [ ]:
import numpy as np
import pytest

# --- Pytest Style Test Functions ---

def test_fermi_at_fermi_level():
    """At E = Ef, the occupancy probability must always be exactly 0.5."""
    assert fermi_dirac(E=3.0, Ef=3.0, T=300) == pytest.approx(0.5)


def test_fermi_excited_state():
    """At E = 3.1 eV, f(E) should be roughly 0.0205 (2.05% occupancy)."""
    expected_probability = 0.020468
    assert fermi_dirac(E=3.1, Ef=3.0, T=300) == pytest.approx(expected_probability, abs=1e-4)


def test_fermi_lower_state():
    """At E = 2.9 eV, f(E) should be roughly 0.9795 (97.95% occupancy)."""
    expected_probability = 0.979531
    assert fermi_dirac(E=2.9, Ef=3.0, T=300) == pytest.approx(expected_probability, abs=1e-4)


@pytest.mark.parametrize("E, Ef, T, expected", [
    (2.5, 3.0, 0, 1.0),  # Below Fermi level at absolute zero -> 100% full
    (3.5, 3.0, 0, 0.0),  # Above Fermi level at absolute zero -> 100% empty
    (3.0, 3.0, 0, 0.5),  # At Fermi level at absolute zero -> 50%
])
def test_absolute_zero_edge_cases(E, Ef, T, expected):
    """Ensures boundary conditions hold perfectly at 0 Kelvin."""
    assert fermi_dirac(E, Ef, T) == expected
